# Tutorial — biblioteca `waveplate`

Este notebook mostra, passo a passo, como controlar as lâminas motorizadas pelo Python.

São dois módulos:

| Módulo | Para quê |
|---|---|
| [waveplate.py](waveplate.py) | biblioteca principal: um objeto `WavePlate` por eixo, com calibração de velocidade e aceleração |
| [arduino_elliptec.py](arduino_elliptec.py) | mesma interface da Thorlabs Elliptec (`EasyMotor`), para usar os códigos do `bell-measurements-main` sem mudar o meio deles |

**Como funciona:** o Arduino gera os pulsos dos motores, a aceleração e a busca do zero. O Python só manda comandos pela serial ("eixo X, vá para 45°") e espera as respostas.

**Unidades:** ângulos em **graus da lâmina**, velocidades em **°/s**, aceleração em **°/s²**.

**Antes de começar:**
1. O firmware atualizado precisa estar gravado no Arduino.
2. Feche o `waveplate_cli.py` e o Monitor Serial do PlatformIO: só um programa por vez pode usar a porta.
3. Rode as células **em ordem**, de cima para baixo.

> **Parar no meio de um movimento:** use o botão ■ *Interrupt* do Jupyter. A biblioteca recebe isso como Ctrl+C e para a lâmina com desaceleração.

## 1. Instalação

Rode uma vez. O `pip install -e` registra a pasta do projeto no Python, para que `import waveplate` e `import arduino_elliptec` funcionem de **qualquer pasta** (inclusive de dentro do `bell-measurements-main`). Edições nos arquivos valem na hora, sem reinstalar — só reinicie o kernel.

In [ ]:
#%pip install pyserial
#%pip install -e .

Obtaining file:///C:/Users/LOQ/Documents/PlatformIO/Projects/lamina_motorizada
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for arduino-waveplate (pyproject.toml): started
  Building editable for arduino-waveplate (pyproject.toml): finished with status 'done'
  Created wheel for arduino-waveplate: filename=arduino_waveplate-0.1.0-0.editable-py3-none-any.whl size=2969 sha256=5a95b0c9e38fb29ead8b9dfb17338457abe801067461707945b451eeb1e0bb90
  Stored in directory: C:\Users\LOQ\AppData\Local\Temp\pip-ephem-wheel-cache-vhjuq02e\w

## 2. Conectar

Abrir a porta **reinicia o Arduino** — a conexão espera ele mandar `READY` (uns 2 s).

Se não souber a porta, a célula abaixo lista as portas seriais disponíveis.

In [1]:
from serial.tools import list_ports

for p in list_ports.comports():
    print(p.device, "-", p.description)

COM4 - USB-SERIAL CH340 (COM4)


In [1]:
import time
from waveplate import WavePlate, WavePlateError

PORT = "COM4"

plate = WavePlate(PORT, "X")
platey = WavePlate(PORT, "Y")
platez = WavePlate(PORT, "Z")
platea = WavePlate(PORT, "A")
plate.disable_when_idle(True, delay_ms=90)
platey.disable_when_idle(True, delay_ms=90)
platez.disable_when_idle(True, delay_ms=90)
platea.disable_when_idle(True, delay_ms=90)
print("Conectado:", platea)

Conectado: WavePlate('COM4', 'A')


> Num script `.py` o jeito recomendado é usar `with`, que fecha tudo sozinho no final (mesmo se der erro):
> ```python
> with WavePlate("COM4") as plate:
>     plate.move_to(45)
> ```
> No notebook usamos `plate = WavePlate(...)` para a conexão continuar aberta entre as células — e fechamos manualmente na última seção.

## 3. Ver a configuração atual

`get_settings()` pergunta ao Arduino os valores em uso. O resultado é um objeto com campos que você pode ler individualmente.

In [2]:
s = platey.get_settings()
print(s)
print()
print("Velocidade mínima :", round(s.min_speed, 2), "°/s")
print("Velocidade máxima :", round(s.max_speed, 2), "°/s")
print("Aceleração        :", round(s.acceleration, 2), "°/s²")
print("Teto do Arduino   :", round(s.speed_limit, 2), "°/s")
print("Limites de ângulo :", s.min_angle, "a", s.max_angle)
print("Resolução         :", round(1 / s.steps_per_degree, 4), "° por passo")

velocidade 54.00 a 216.00 °/s (máx. possível 225.00 °/s), aceleração 360.00 °/s², limites -3600° a 3600°, driver sempre habilitado, 22.2222 passos/°

Velocidade mínima : 54.0 °/s
Velocidade máxima : 216.0 °/s
Aceleração        : 360.0 °/s²
Teto do Arduino   : 225.0 °/s
Limites de ângulo : -3600.0 a 3600.0
Resolução         : 0.045 ° por passo


## 4. Fim de curso e zero mecânico

Primeiro, teste o sensor: rode a célula com ele livre e depois apertando-o com a mão.

In [15]:
print("Fim de curso:", "ACIONADO" if plate.limit_switch_triggered() else "livre")

Fim de curso: livre


`home()` gira até achar o fim de curso, recua e se aproxima devagar (para o ponto ser repetível) e define ali o **zero mecânico**. A função só retorna quando terminar.

Se não houver sensor, use `set_zero()`: a posição atual passa a ser 0°, sem mexer o motor.

In [54]:
zero = plate.home()
print(f"Referenciado em {zero:.2f}°")

# Alternativa sem sensor:
# plate.set_zero()

Referenciado em 0.00°


## 5. Movimentos básicos

| Função | O que faz |
|---|---|
| `move_to(angle)` | vai até o ângulo **absoluto** |
| `move_by(delta)` | anda `delta` graus a partir de onde está (**relativo**) |
| `get_position()` | retorna o ângulo atual |

Por padrão as duas primeiras **esperam o movimento terminar** e retornam a posição final.

In [12]:
print("move_to(90)   →", platez.move_to(90))
print("move_to(-45)  →", platez.move_to(-45))
print("move_by(+30)  →", platez.move_by(30))
print("move_by(-15)  →", platez.move_by(-15))
print("get_position()→", platez.get_position())
print("get_position()→", platez.get_position())

move_to(90)   → 90.0
move_to(-45)  → -45.0
move_by(+30)  → -14.99
move_by(-15)  → -30.02
get_position()→ -30.02
get_position()→ -30.02


In [5]:
print("move_to(90)   →", plate.move_to(90))
print("move_to(90)   →", platey.move_to(90))
print("move_to(-45)  →", plate.move_to(-45))
print("move_to(-45)  →", platey.move_to(-45))
print("move_by(+30)  →", plate.move_by(30))
print("move_by(+30)  →", platey.move_by(30))
print("move_by(-15)  →", plate.move_by(-15))
print("move_by(-15)  →", platey.move_by(-15))
print("get_position()→", plate.get_position())
print("get_position()→", platey.get_position())

move_to(90)   → 90.0
move_to(90)   → 90.0
move_to(-45)  → -45.0
move_to(-45)  → -45.0
move_by(+30)  → -14.99
move_by(+30)  → -14.99
move_by(-15)  → -30.02
move_by(-15)  → -30.02
get_position()→ -30.02
get_position()→ -30.02


**Por que às vezes aparece 35.01 em vez de 35?** O motor anda em passos discretos (veja a resolução na seção 3), então o Arduino vai para o passo mais próximo do ângulo pedido e informa a posição real.

Vários `move_by()` seguidos **não acumulam esse erro**: cada um parte do último ângulo *pedido*, não do arredondado.

In [56]:
platey.move_to(0)
for _ in range(10):
    platey.move_by(0.5)
print("Depois de 10 × 0.5°:", platey.get_position())   # ≈ 5.00

Depois de 10 × 0.5°: 5.0


## 6. Varredura de ângulos (uso típico em experimento)

Padrão comum: ir para cada ângulo, esperar estabilizar, medir, guardar. Troque `measure()` pela leitura do seu instrumento (power meter, Time Tagger, câmera…).

In [61]:
def measure():
    """Coloque aqui a leitura do seu instrumento."""
    return 0.0

SETTLE_TIME_S = 0.2          # espera a lâmina parar de vibrar antes de medir
angles = range(0, 181, 15)   # 0, 15, 30, ..., 180

results = []
for angle in angles:
    actual = platex.move_to(angle)
    time.sleep(SETTLE_TIME_S)
    value = measure()
    results.append((actual, value))
    print(f"{actual:7.2f}°  →  {value}")

platey.move_to(0)

KeyboardInterrupt: 

## 7. Movimento sem bloquear

Com `wait=False`, `move_to()` retorna **na hora** (com o ângulo-alvo real) e o Python fica livre enquanto a lâmina gira — útil para ler um sensor *durante* o movimento ou mostrar progresso.

- `is_moving()` → `True` enquanto não terminou
- `get_position()` → funciona durante o movimento
- `wait_move()` → bloqueia até terminar e retorna a posição final
- `stop()` → para com desaceleração

`home(wait=False)` funciona do mesmo jeito.

Enquanto a lâmina se move, comandos que exigem parada (`move_to`, `configure`, `home`…) dão `WavePlateError`.

In [62]:
target = plate.move_to(360, wait=False)
print("Indo para", target)

while plate.is_moving():
    print(f"  posição: {plate.get_position():7.2f}°")
    time.sleep(0.25)

print("Chegou em", plate.wait_move())

Indo para 360.0
  posição:    0.22°
  posição:   25.96°
  posição:   74.20°
  posição:  129.06°
  posição:  183.87°
  posição:  238.73°
  posição:  293.58°
  posição:  339.12°
Chegou em 360.0


In [63]:
# Parando no meio do caminho
plate.move_to(0, wait=False)
time.sleep(1.0)
print("Parou em", plate.stop())

Parou em 118.44


## 8. Calibrar velocidade e aceleração

O perfil de movimento é **trapezoidal com velocidade mínima**:

```
velocidade
  max_speed ┤      ┌──────────┐
            │     /            \        inclinação = acceleration
  min_speed ┤ ───┘              └───
            └──────────────────────── tempo
```

- **`min_speed`**: o motor arranca e chega direto nessa velocidade (pula a faixa lenta, onde vibra). Também é a velocidade da busca do zero.
- **`max_speed`**: velocidade de cruzeiro. Não passa de `speed_limit` (o Arduino não gera pulsos mais rápido).
- **`acceleration`**: quão rápido sai de `min_speed` para `max_speed`. Alta demais → o motor perde passos (a posição fica errada sem aviso).

`configure()` só muda o que você passar e retorna a configuração aplicada. Cada eixo tem a sua.

In [ ]:
print("Antes  :", plate.get_settings())

plate.configure(min_speed=20, max_speed=90, acceleration=150)
print("Depois :", plate.get_settings())

plate.configure(max_speed=120)   # muda só a máxima
print("Só max :", plate.get_settings())

### Teste de calibração: medir o tempo de um movimento

Rode com valores diferentes e compare tempo, ruído e vibração. Para conferir se **não perdeu passos**: depois de várias idas e voltas, rode `home()` e veja se o zero continua no mesmo lugar físico (ou faça uma marca na lâmina).

In [ ]:
def time_move(start, end):
    plate.move_to(start)
    t0 = time.perf_counter()
    plate.move_to(end)
    return time.perf_counter() - t0

for vmax, accel in [(60, 100), (120, 200), (200, 400)]:
    plate.configure(max_speed=vmax, acceleration=accel)
    t = time_move(0, 360)
    print(f"max_speed={vmax:4} °/s  acceleration={accel:4} °/s²  →  360° em {t:.2f} s")

plate.move_to(0)

Os ajustes ficam só na memória do Arduino — **somem quando ele reinicia** (por exemplo, ao reconectar). Para voltar aos valores de fábrica sem reiniciar:

In [ ]:
print(plate.restore_defaults())

Quando achar bons valores, tem duas opções:
1. chamar `plate.configure(...)` logo depois de conectar em todos os seus códigos; ou
2. colocar como padrão no firmware (`VEL_MIN_PADRAO`, `VEL_MAX_PADRAO`, `ACEL_PADRAO` em `src/main.cpp` — lá ficam em passos inteiros/s; multiplique °/s por `steps_per_degree` e divida pelo microstep).

## 9. Driver: segurar ou soltar o motor

| Modo | Vantagem | Desvantagem |
|---|---|---|
| sempre habilitado (`False`) | segura a posição com força e precisão | motor esquenta e pode chiar |
| desabilitado quando parado (`True`) | motor frio e silencioso | ao soltar, o rotor pode "cair" até ~meio passo inteiro (≈0,18° na lâmina) e o Arduino não percebe |

No modo `True` o driver desliga `delay_ms` depois de cada movimento e religa sozinho no próximo.

> No CNC Shield o pino ENABLE é **um só para todos os eixos**: o driver só desliga de fato quando todos os eixos pedirem para desligar. **Para medições, prefira sempre habilitado** — a lâmina não pode se mexer enquanto você integra.

In [ ]:
plate.disable_when_idle(True, delay_ms=200)
plate.move_to(45)
print("Agora o motor solta 0,2 s depois de parar. Tente girar com a mão.")

In [ ]:
plate.disable_when_idle(False)
plate.enable()      # liga já, sem esperar o próximo movimento
print("Motor segurando a posição.")

# Também dá para soltar/segurar manualmente a qualquer momento:
# plate.disable()
# plate.enable()

Dá para escolher o modo já ao conectar: `WavePlate("COM4", disable_when_idle=True)`.

## 10. Vários eixos

Cada eixo do CNC Shield (X, Y, Z, A) é uma lâmina. Os eixos ligados estão em `EIXOS_CFG` no firmware. Vários objetos na **mesma porta** compartilham a conexão.

`open_plates(PORT)` abre todos os eixos do firmware de uma vez e retorna um dicionário `{letra: WavePlate}`. Como a porta já está aberta (variável `plate`), a conexão é reaproveitada e o Arduino não reinicia.

In [ ]:
from waveplate import open_plates

plates = open_plates(PORT)
print("Eixos:", list(plates))

# Os dois andam ao mesmo tempo com wait=False
plates["X"].move_to(90, wait=False)
plates["Y"].move_to(45, wait=False)
print("X chegou em", plates["X"].wait_move())
print("Y chegou em", plates["Y"].wait_move())

for p in plates.values():
    p.close()

## 11. Usando no lugar da Thorlabs Elliptec

O módulo `arduino_elliptec` imita o `EasyMotor` / `ElliptecMotor` usados no `bell-measurements-main`. Cada **endereço** Elliptec (`addr`) vira um **eixo** do Arduino:

| Thorlabs | Arduino |
|---|---|
| `from EASYMOTOR import EasyMotor` | `from arduino_elliptec import EasyMotor` |
| `stage = EasyMotor(PORT)` | `stage = EasyMotor(PORT, axes={mountA: "Y", mountB: "X"})` |
| `stage.home(addr=[mountB, mountA])` | igual (os dois fazem a referência **ao mesmo tempo**) |
| `stage.move_to(angle, addr=mountA)` | igual |
| `stage.close()` | igual (também desliga os drivers) |

Sem `axes`, vale o mapa padrão **0→X, 1→Y, 2→Z, 3→A**. Com Bob no X e Alice no Z (e o Z ligado no firmware), `EasyMotor(PORT)` funciona sem mudar nem o construtor.

Outras funções no estilo Elliptec: `move_by`, `get_position`, `stop`, `is_moving`, `wait_move`, `get_connected_addrs`, `set_default_addr`. `addr` aceita um número, uma lista ou `"all"`.

> **Recalibre os ângulos.** O zero agora é onde o fim de curso aciona, não o zero de fábrica da Thorlabs: `H_alice`, `H_bob` etc. precisam ser medidos de novo.

In [ ]:
from arduino_elliptec import EasyMotor

mountA = 2      # Alice
mountB = 0      # Bob

plate.close()   # libera o eixo X aberto nas seções anteriores

stage = EasyMotor(PORT, axes={mountA: "Y", mountB: "X"})
print(stage)

stage.home(addr=[mountB, mountA])
stage.move_to(100, addr=mountA)
stage.move_to(146.84, addr=mountB)
print("Alice:", stage.get_position(addr=mountA), " Bob:", stage.get_position(addr=mountB))

Para calibrar velocidade e aceleração de um endereço, pegue o `WavePlate` dele com `stage.plate(addr)`:

In [ ]:
stage.plate(mountA).configure(max_speed=120, acceleration=200)
print("Alice:", stage.plate(mountA).get_settings())
print("Bob  :", stage.plate(mountB).get_settings())

## 12. Tratando erros

| Exceção | Quando acontece |
|---|---|
| `ValueError` | valor inválido detectado no Python (ângulo fora dos limites, `max_speed` acima do teto, endereço sem eixo…) |
| `WavePlateError` | o Arduino recusou (ex.: comando durante movimento, fim de curso não encontrado) |
| `TimeoutError` | o Arduino não respondeu (cabo, porta, firmware travado) |
| `KeyboardInterrupt` | você interrompeu — a lâmina já foi parada antes da exceção chegar até você |

In [ ]:
try:
    stage.move_to(99999, addr=mountA)
except ValueError as e:
    print("ValueError:", e)

try:
    stage.move_to(10, addr=1)          # endereço sem eixo configurado
except ValueError as e:
    print("ValueError:", e)

try:
    stage.plate(mountA).configure(max_speed=10_000)
except ValueError as e:
    print("ValueError:", e)

## 13. Encerrar

`close()` para as lâminas (se estiverem andando), **desliga os drivers** e libera a porta serial. Rode sempre ao terminar — senão a porta fica presa e outros programas não conseguem abrir (se acontecer, reinicie o kernel).

Para fechar mantendo os motores segurando a posição: `stage.close(disable=False)`.

In [ ]:
stage.close()
print("Conexão encerrada.")

## Resumo rápido

```python
from waveplate import WavePlate, WavePlateError, open_plates

with WavePlate("COM4", "X", disable_when_idle=False) as plate:
    plate.configure(min_speed=20, max_speed=90, acceleration=150)  # opcional
    plate.home()                       # zero mecânico

    plate.move_to(45)                  # absoluto (espera)
    plate.move_by(-10)                 # relativo (espera)
    plate.get_position()               # onde está

    plate.move_to(180, wait=False)     # sem bloquear
    while plate.is_moving(): ...
    plate.wait_move()                  # ou stop()

    plate.get_settings()               # valores atuais
    plate.restore_defaults()
    plate.limit_switch_triggered()     # True/False
    plate.enable(); plate.disable()
    plate.disable_when_idle(True)

# Código da Thorlabs:
from arduino_elliptec import EasyMotor
stage = EasyMotor("COM4", axes={2: "Y", 0: "X"})
stage.home(addr=[0, 2])
stage.move_to(45, addr=2)
stage.close()
```